In [ ]:
import datetime
import matplotlib.pyplot as plt
import numpy as np
import scipy
import time 
import pandas as pd
import statsmodels as sm
import tqdm
import astropy.units as u

from sklearn.model_selection import GroupShuffleSplit
from sklearn import metrics
from pathlib import Path

import sys
localpath = Path('/Users/mrutala/projects/ASWEstimator/code/')
sys.path.append(localpath.as_posix())
import ASWEstimator
import performance 

First we'll set up some basic parameters for the experiment overall: the start and stop dates; the duration and buffer (padding on each end) of the ICMEs; the interpolation buffer (how much data is averaged on each end for the linear interpolation); the number of splits; the number of samples to draw; and a directory for all figures and files to be saved in.

In [ ]:
start = datetime.datetime(2010, 1, 1)
stop = datetime.datetime(2015, 1, 1)

icme_duration   = 3.75 * u.day # conservative duration (Richardson & Cane 2010)
icme_buffer     = 0.25 * u.day # onservative duration (Richardson & Cane 2010)
interp_buffer   = 1.0 * u.day # how much to use in the interpolation

n_splits = 10

num_samples = 100

experiment_dir = localpath / 'experiments/1DGPR/'

Get background and transient data

In [ ]:
inputs = ASWEstimator.ASWEstimator(start, stop, rmax=1.1, latmax=20)
inputs._icme_duration = icme_duration # conservative duration (Richardson & Cane 2010)
inputs._icme_duration_buffer = icme_buffer # conservative buffer (Richardson & Cane 2010)
inputs._icme_interp_buffer = interp_buffer

inputs.getSolarWind()
inputs.getTransients()

In [ ]:
# %%
# =============================================================================
# Add fake ICMEs
# =============================================================================

split_bool_columns = pd.MultiIndex.from_product((inputs.boundarySources, 
                                                     ['test', 'train', 'icme']))
split_bool_df = pd.DataFrame(data = False, 
                             index = inputs.solar_wind.index,
                             columns = split_bool_columns)

split_dfs = [split_bool_df.copy() for _ in range(n_splits)] 

for source in inputs.boundarySources:
    
    df = inputs.solar_wind[source].copy()
    df['mjd'] = inputs.solar_wind['mjd']
    
    # Define ICME-sized groups
    group_length = (icme_duration + 2*icme_buffer).to(u.day).value # days
    groups = pd.Series(
        index   = df.index, 
        data    = np.floor((df['mjd'] - df['mjd'].iloc[0])/group_length)
        )
    
    # Exclude ICMEs from the train/test split
    groups[df.query("ICME == True").index] = np.nan
    
    # Perform Group-KFold split, leaving ICMES out
    from sklearn.model_selection import GroupKFold
    group_kfold = GroupKFold(n_splits=n_splits, shuffle=True)
    nonICME_index = df.query("ICME == False").index
    
    for i, (train, test) in enumerate(group_kfold.split(df.loc[nonICME_index,:], groups=groups[nonICME_index])):
        
        split_dfs[i].loc[nonICME_index[train], (source, 'train')] = True
        split_dfs[i].loc[nonICME_index[test], (source, 'test')] = True
        split_dfs[i].loc[df.query("ICME == True").index, (source, 'icme')] = True





In [ ]:
# Visualize training and test sets
fig, axs = plt.subplots(nrows=len(inputs.boundarySources), figsize=(6,4), sharex=True)
for j, source in enumerate(inputs.boundarySources):
    axs[j].set(ylabel = source, xlim=[inputs.solar_wind['mjd'].min(), inputs.solar_wind['mjd'].max()])
    for i, split_df  in enumerate(split_dfs):
        
        axs[j].fill_between(inputs.solar_wind['mjd'], 
                            i+1-0.33, i+1+0.33, where=split_df[(source, 'train')],
                            color='xkcd:cornflower', lw=0)
        axs[j].fill_between(inputs.solar_wind['mjd'], 
                            i+1-0.33, i+1+0.33, where=split_df[(source, 'test')],
                            color='xkcd:purple', lw=0)
        axs[j].fill_between(inputs.solar_wind['mjd'], 
                            i+1-0.33, i+1+0.33, where=split_df[(source, 'icme')],
                            color='xkcd:red', lw=0)

      
fig.supylabel('Split Number')
fig.supxlabel('MJD')
    

plt.savefig(experiment_dir / 'Train_Test_split.png')

In [ ]:
# Compare our train/test split to the real background/ICME split
elapsed_time_ICME = []
elapsed_time_test = []
for s_df in split_dfs:
    for source in inputs.boundarySources:
        # Gaps caused by ICMEs
        elapsed_time_ICME.extend(np.diff(inputs.solar_wind['mjd'][~s_df[(source, 'icme')]]))
        elapsed_time_test.extend(np.diff(inputs.solar_wind['mjd'][s_df[(source, 'train')]]))
        
        
fig, ax = plt.subplots()
h_ICME, bins = np.histogram(elapsed_time_ICME, bins=np.arange(0,15,1))
h_test, bins = np.histogram(elapsed_time_test, bins=bins)

ax.stairs(h_ICME, bins, label='True', fill=False, lw=1)
ax.stairs(h_test, bins, label='Simulated', fill=False, lw=1)

KS_results = scipy.stats.ks_2samp(elapsed_time_ICME, elapsed_time_test)

ax.set(xlabel='Time Since Background Solar Wind [days]', 
       ylabel='#', yscale='log')
ax.annotate("K-S p-value: {:.3f}".format(KS_results.pvalue), (0,1), (1,-1), 
            'axes fraction', 'offset fontsize')

plt.savefig(experiment_dir / 'Train_Test_split_stats.png')

Now we'll train a GP and LI model for each train/test split. Prior to running the models, we can check whether they ahve already been run to save time.

In [ ]:
# Save a copy of the original input for comparison
inputs_ref = inputs.copy()

# Perform modeling
for i, s_df in enumerate(split_dfs):
    
    # Copy the original inputs for the Gaussian process, and add test ICMEs
    GP_input = inputs.copy()
    for source in GP_input.boundarySources:
        GP_input.solar_wind.loc[:, (source, 'ICME')] = s_df.loc[:, (source, 'icme')] | s_df.loc[:, (source, 'test')]
        
    # Copy the GP input again for the Linear Interpolation
    LI_input = GP_input.copy()
    
    GP_input.makeBackgroundDistribution(GP = True, n_samples = num_samples)
    GP_input.save(experiment_dir / '{0:02d}_GP_input.pkl'.format(i))
        
    # Generate the background distribution, removing the fake ICMEs
    LI_input.makeBackgroundDistribution(interpolate = True, n_samples = num_samples)
    LI_input.save(experiment_dir / '{0:02d}_LI_input.pkl'.format(i))

    # Finally, write the split df to a file as well
    s_df.to_csv(experiment_dir / '{0:02d}_split.csv'.format(i))

    